In [ ]:
import os
from dotenv import load_dotenv
from copy import deepcopy
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [1]:
import vllm
import torch
from transformers import AutoTokenizer

# --- 1. Load the vLLM engine and the model's tokenizer ---
# The tokenizer contains the chat template.
MODEL_NAME = "Qwen/Qwen3-0.6B"

llm = vllm.LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16
)

# It's crucial to load the tokenizer for the specific model you are using.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# --- 2. Define your conversation in a structured list ---
# This is a clean, model-agnostic way to represent a conversation.
messages = [
    {"role": "system", "content": "You are a helpful assistant based in Bandung, Indonesia."},
    {"role": "user", "content": "What are some good places to visit nearby for a weekend trip?"}
]

# --- 3. Apply the chat template to format the prompt ---
# This is the key step. apply_chat_template does all the heavy lifting.
# - tokenize=False: Returns a formatted string, which is what vLLM needs.
# - add_generation_prompt=True: Adds the tokens to signal the start of the assistant's turn (e.g., "<|im_start|>assistant\n").
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Let's see what the formatted prompt looks like:
print("--- Formatted Prompt for vLLM ---")
print(prompt)
print("-" * 35)

# --- 4. Use the formatted prompt with vLLM ---
sampling_params = vllm.SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=500,
    stop=["<|im_end|>"]
)

outputs = llm.generate([prompt], sampling_params)

# --- 5. Print the result ---
for output in outputs:
    generated_text = output.outputs[0].text
    print("--- Model Response ---")
    print(generated_text.strip())

/home/hanif_zhafran07/absa-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 02-20 06:45:14 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': torch.bfloat16, 'disable_log_stats': True}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-20 06:45:14 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 02-20 06:45:14 [model.py:1561] Using max model len 40960


2026-02-20 06:45:15,336	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


...

(EngineCore_DP0 pid=41009) INFO 02-20 06:46:33 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=41009) INFO 02-20 06:46:42 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 19.66 s
(EngineCore_DP0 pid=41009) INFO 02-20 06:46:42 [monitor.py:34] torch.compile takes 30.21 s in total
(EngineCore_DP0 pid=41009) INFO 02-20 06:46:43 [gpu_worker.py:356] Available KV cache memory: 17.2 GiB
(EngineCore_DP0 pid=41009) INFO 02-20 06:46:43 [kv_cache_utils.py:1307] GPU KV cache size: 161,008 tokens
(EngineCore_DP0 pid=41009) INFO 02-20 06:46:43 [kv_cache_utils.py:1312] Maximum concurrency for 40,960 tokens per request: 3.93x


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 18.46it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 19.85it/s]


(EngineCore_DP0 pid=41009) INFO 02-20 06:46:49 [gpu_model_runner.py:5063] Graph capturing finished in 6 secs, took 0.47 GiB
(EngineCore_DP0 pid=41009) INFO 02-20 06:46:49 [core.py:272] init engine (profile, create kv cache, warmup model) took 42.35 seconds
INFO 02-20 06:46:51 [llm.py:343] Supported tasks: ['generate']
--- Formatted Prompt for vLLM ---
<|im_start|>system
You are a helpful assistant based in Bandung, Indonesia.<|im_end|>
<|im_start|>user
What are some good places to visit nearby for a weekend trip?<|im_end|>
<|im_start|>assistant

-----------------------------------


Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.09s/it, est. speed input: 12.32 toks/s, output: 162.14 toks/s]

--- Model Response ---
<think>
Okay, the user is asking for good weekend places to visit near Bandung. First, I need to make sure I know Bandung's major attractions. Let me recall. Bandung is known for its historical sites, like the National Museum of History, the Palace of the Sultan, and the Bumi Permai park. There's also the Konung General Prabowo Square, which is a big public square. 

Then there's the food scene. The Bumi Permai park is famous for its local cuisine, so that's a good spot to eat. The historical areas around the palace and the museum are popular. Maybe mention the MRT stations as well, since they're central to the city.

I should also consider the best time to visit. Weekends are usually busy, so maybe suggest visiting in the afternoon or early evening. Also, maybe include some cultural spots like the Bumi Permai Park or the Konung General Prabowo Square. Are there any other attractions? The Bandung Cultural Center might be a good addition. 

Wait, the user is from 